# 4.1 · Standardising an ontology language

Before OWL there were many ontology languages (OBO, F-logic, KL-ONE…), causing interoperability problems. The W3C standardised **OWL** (2004), influenced by SHOE, DAML-ONT, OIL, DAML+OIL and 20 years of DL research.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))  # repo paths
sys.path.insert(0, '.')
import matplotlib; matplotlib.use('Agg')
import ch4_toolkit as ch4
import owlready2, rdflib, owlrl, pandas as pd
print('owlready2', owlready2.VERSION, '| rdflib', rdflib.__version__, '| toolkit ready')

## 4.1.1 How to design an ontology language (Figure 4.1)
An iterative 7-step process. Here it is as a Python data structure you can inspect.

In [ ]:
for s in ch4.LANGUAGE_DESIGN_PROCESS:
    print(f"Step {s['step']}: {s['name']}")
    for t in s['tasks']:
        print('   ', t)

**OWL's stated design goals** (roughly steps 1–3 of the process):

In [ ]:
for i, g in enumerate(ch4.OWL_DESIGN_GOALS, 1):
    print(f'{i}. {g}')

## 4.1.2 What makes OWL different from a plain DL?

In [ ]:
for d in ch4.OWL_VS_DL:
    print('•', d)

## 4.1.3 The OWL family, first version (OWL 1 species)

Three species: **OWL Lite** (`SHIF(D)`), **OWL DL** (`SHOIN(D)`), **OWL Full** (not a DL). Lite/DL have a model-theoretic semantics; Full has RDF freedom and is undecidable.

In [ ]:
for name, info in ch4.OWL1_SPECIES.items():
    print(f"{name}  —  DL: {info['dl']}  (is a DL: {info['is_dl']})")
    print('   ', info['notes'])
    print('    features:', ', '.join(info['features'][:4]), '...')


### Table 4.1 — OWL class constructs ↔ DL ↔ example
Rendered as a DataFrame, and then **built for real** with `owlready2`.

In [ ]:
ch4.table_4_1_constructs()

### Table 4.2 — OWL axioms ↔ DL ↔ example

In [ ]:
ch4.table_4_2_axioms()

### Building those constructs/axioms in Python
`build_construct_demo()` creates `Man ≡ Human ⊓ Male`, `Professional ≡ Doctor ⊔ Lawyer`, `∀hasChild.Doctor`, `≥2 hasChild`, `Male ⊑ ¬Female`, `Human ⊑ Animal ⊓ Biped`, etc. — and we render them back in DL notation.

In [ ]:
demo = ch4.build_construct_demo()
for cls in demo.classes():
    print(ch4.dl_render(cls))


## Example 4.1 — The African Wildlife Ontology (AWO)

The book's tutorial ontology: 10 classes (Lion, Giraffe, Plant…), object properties `eats` and `is-part-of`, and the axiom *“giraffes eat only leaves”* (`Giraffe ⊑ ∀eats.Leaf`). We build it in Python instead of typing XML.

In [ ]:
awo = ch4.build_awo(level=0)
print('classes:', [c.name for c in awo.classes()])
print()
print(ch4.dl_render(awo.Giraffe))

**Listing 4.1 twin** — the `Giraffe` class serialised to the *required* RDF/XML exchange syntax, generated from the Python ontology:

In [ ]:
print(ch4.awo_giraffe_owl_snippet(awo))

### Extend it (AWO v1) and run the reasoner
Add proper parthood, `Impala`, `Warthog`, `RockDassie`, and herbivore/carnivore/omnivore definitions. The book classifies `Carnivore ⊑ Animal` with HermiT; here the OWL 2 RL reasoner materialises the subclass/again from the `Animal ⊓ …` definitions.

In [ ]:
awo1 = ch4.build_awo(level=1)
print('classes:', len(list(awo1.classes())))
g, b, a, new = ch4.reason_owlrl(awo1)
print(f'closure {b} -> {a} (+{len(new)} inferred triples)')
# show a few inferred rdfs:subClassOf statements
from rdflib import RDFS
subs = [(s, o) for (s, p, o) in new if p == RDFS.subClassOf]
for s, o in list(subs)[:8]:
    print('  inferred subClassOf:', s.split('#')[-1], '⊑', o.split('#')[-1])

> **Note on reasoning.** Full DL classification of, e.g., *which individuals are carnivores* needs a DL reasoner (HermiT). OWL 2 RL (used here) is a sound, scalable rule fragment — perfect for subclass/property propagation, which is what we show.